<a href="https://colab.research.google.com/github/aHawkins63/Feature-Engineering/blob/main/anime_collaborative_filtering_recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q scikit-surprise

     |████████████████████████████████| 11.8 MB 35.0 MB/s 


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
from surprise import Reader, Dataset, SVD
from surprise.model_selection import cross_validate
import surprise

In [ ]:
df_anime = pd.read_csv('../content/drive/MyDrive/anime.csv')
df_rating = pd.read_csv('../content/drive/MyDrive/rating.csv')

In [ ]:
df_anime = df_anime[['anime_id', 'name']]
#df_anime.rename(columns={'uid':'anime_id'}, inplace=True)
df_anime = df_anime.drop_duplicates()

In [ ]:
df_rating.head()

,user_id,anime_id,rating
0,1,20,-1
1,1,24,-1
2,1,79,-1
3,1,226,-1
4,1,241,-1


In [ ]:
df_anime.sample(5)

,anime_id,name
4842,15279,Kira Kira 5th Anniversary Live Anime: Kick Sta...
9902,23515,Otsuki-sama to Oujo
9946,22631,Peeping Life: Tezuka Pro - Tatsunoko Pro Wonde...
7161,5193,Shinjuku Shin-chan Patrol: Warui Otona ni Ki w...
5873,22435,New Mobile Report Gundam Wing: Frozen Teardrop...


In [ ]:
df_rating_cp = df_rating[df_rating['rating']!=-1]
df_rating_cp = df_rating_cp.sample(frac=1)[:1000000].sort_values('user_id')
# df_rating_cp.shape

In [ ]:
data = Dataset.load_from_df(df_rating_cp, Reader(rating_scale=(1,10)))
trainset = data.build_full_trainset()

In [ ]:
import time as time
start = time.time()
model = SVD()
model.fit(trainset)
cross_validate(model, data, measures=["RMSE", "MAE"], cv=5, verbose=True)
end = time.time()
print(end - start)

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    1.2852  1.2824  1.2826  1.2857  1.2840  1.2840  0.0013  
MAE (testset)     0.9810  0.9797  0.9796  0.9826  0.9795  0.9805  0.0012  
Fit time          41.75   42.56   41.34   41.80   43.90   42.27   0.90    
Test time         2.72    2.46    2.72    2.46    2.56    2.58    0.12    
282.27117252349854


In [ ]:
df_rating_cp[df_rating_cp['user_id']==5]

,user_id,anime_id,rating
459,5,5042,1
308,5,24,1
475,5,6347,1
419,5,2397,7
608,5,15059,6
...,...,...,...
379,5,903,1
536,5,10172,1
604,5,14813,3
762,5,30831,8


In [ ]:
model.predict(5, 4063, 2)

Prediction(uid=5, iid=4063, r_ui=2, est=3.67907048747875, details={'was_impossible': False})

In [ ]:
user_id = 5
all_anime = df_rating_cp.anime_id.unique()
watched = df_rating_cp[df_rating_cp.user_id==user_id].anime_id
not_watched = [anime for anime in all_anime if anime not in watched]

In [ ]:
# predict
score = [model.predict(user_id, anime_id) for anime_id in not_watched]
anime_id = []
pred_score = []
for i in range(0, len(score)):
    anime_id.append(score[i].iid)
    pred_score.append(score[i].est)
df_pred = pd.DataFrame({'anime_id':anime_id, 'pred_score':pred_score})

In [ ]:
df_pred_real = df_pred.sort_values('pred_score', ascending=False).head(10)

In [ ]:
df_pred_real.merge(df_anime, how='left', on='anime_id')

,anime_id,pred_score,name
0,11577,8.775455,Steins;Gate Movie: Fuka Ryouiki no Déjà vu
1,205,8.515486,Samurai Champloo
2,9253,8.348355,Steins;Gate
3,7724,8.348279,Shiki
4,7311,8.316361,Suzumiya Haruhi no Shoushitsu
5,11741,8.184633,Fate/Zero 2nd Season
6,44,7.808909,Rurouni Kenshin: Meiji Kenkaku Romantan - Tsui...
7,5341,7.773150,Ookami to Koushinryou II
8,16498,7.730372,Shingeki no Kyojin
9,777,7.723013,Hellsing Ultimate


In [10]:
import zipfile
import os
import pandas as pd
from surprise import Reader, Dataset, SVD

with zipfile.ZipFile('/content/animedata.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/animedata')

anime_path = '/content/animedata/anime.csv'
rating_path = '/content/animedata/rating.csv'

df_anime = pd.read_csv(anime_path)
df_rating = pd.read_csv(rating_path)

df_anime = df_anime[['anime_id', 'name']].drop_duplicates()
df_rating_cp = df_rating[df_rating['rating'] != -1]
df_rating_cp = df_rating_cp.sample(frac=1, random_state=42)[:1000000].sort_values('user_id')

reader = Reader(rating_scale=(1, 10))
data = Dataset.load_from_df(df_rating_cp[['user_id', 'anime_id', 'rating']], reader)
trainset = data.build_full_trainset()
model = SVD(random_state=42)
model.fit(trainset)

all_anime = df_rating_cp.anime_id.unique()
for user_id in [7, 9, 11]:
    watched = df_rating_cp[df_rating_cp.user_id == user_id].anime_id.values
    not_watched = [anime for anime in all_anime if anime not in watched]

    score = [model.predict(user_id, anime_id) for anime_id in not_watched]
    anime_id_list = [s.iid for s in score]
    pred_score_list = [s.est for s in score]

    df_pred = pd.DataFrame({'anime_id': anime_id_list, 'pred_score': pred_score_list})
    df_pred_real = df_pred.sort_values('pred_score', ascending=False).head(10)

    print(f'\nRecommendations for User ID: {user_id}')
    display(df_pred_real.merge(df_anime, how='left', on='anime_id'))


Recommendations for User ID: 7


,anime_id,pred_score,name
0,820,9.405298,Ginga Eiyuu Densetsu
1,164,9.346164,Mononoke Hime
2,5114,9.156172,Fullmetal Alchemist: Brotherhood
3,9253,9.073346,Steins;Gate
4,6547,9.054904,Angel Beats!
5,4282,9.053923,Kara no Kyoukai 5: Mujun Rasen
6,15417,9.043709,Gintama&#039;: Enchousen
7,11577,9.035752,Steins;Gate Movie: Fuka Ryouiki no Déjà vu
8,24701,9.032044,Mushishi Zoku Shou 2nd Season
9,32281,9.017324,Kimi no Na wa.



Recommendations for User ID: 9


,anime_id,pred_score,name
0,820,9.964791,Ginga Eiyuu Densetsu
1,28977,9.560931,Gintama°
2,32281,9.530793,Kimi no Na wa.
3,5114,9.440124,Fullmetal Alchemist: Brotherhood
4,9253,9.412206,Steins;Gate
5,9969,9.349934,Gintama&#039;
6,32935,9.339499,Haikyuu!!: Karasuno Koukou VS Shiratorizawa Ga...
7,15417,9.331775,Gintama&#039;: Enchousen
8,918,9.298258,Gintama
9,19,9.296334,Monster



Recommendations for User ID: 11


,anime_id,pred_score,name
0,820,9.628047,Ginga Eiyuu Densetsu
1,5114,9.424019,Fullmetal Alchemist: Brotherhood
2,11981,9.252126,Mahou Shoujo Madoka★Magica Movie 3: Hangyaku n...
3,19,9.220899,Monster
4,1,9.215185,Cowboy Bebop
5,2001,9.156399,Tengen Toppa Gurren Lagann
6,627,9.115283,Major S1
7,32281,9.107579,Kimi no Na wa.
8,11061,9.086877,Hunter x Hunter (2011)
9,877,9.084567,Nana
